In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

In [18]:
# === Load dataset ===
df = pd.read_csv("End-user-bottleneck-dataset.csv")

# === Drop timestamp ===
df.drop(columns=["timestamp"], inplace=True)

# === Normalize label column (lowercase all labels) ===
df["label"] = df["label"].str.strip().str.lower()

df.info()
df['label'].value_counts()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2029 entries, 0 to 2028
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   avg_total_cpu         2029 non-null   float64
 1   avg_per_core          2029 non-null   float64
 2   avg_ram_percent       2029 non-null   float64
 3   avg_swap_percent      2029 non-null   float64
 4   avg_av_cpu            2029 non-null   float64
 5   avg_network_proc_cpu  2029 non-null   float64
 6   avg_tcp_retrans_rate  2029 non-null   float64
 7   label                 2029 non-null   object 
dtypes: float64(7), object(1)
memory usage: 126.9+ KB


,avg_total_cpu,avg_per_core,avg_ram_percent,avg_swap_percent,avg_av_cpu,avg_network_proc_cpu,avg_tcp_retrans_rate
count,2029.000000,2029.000000,2029.000000,2029.000000,2029.000000,2029.000000,2029.000000
mean,1.046889,1.069800,1.005445,2.065887,1.222693,0.932249,1.573758
std,0.706130,0.672374,0.121350,1.656704,3.077226,1.526453,2.670505
min,0.000000,0.000000,0.401969,0.174677,0.000000,0.000000,0.000000
25%,0.670941,0.750072,0.982146,0.988248,0.000000,0.000000,0.000000
50%,0.937960,0.989603,1.003975,1.269800,0.407038,0.858000,0.068179
75%,1.215240,1.212278,1.040067,2.970041,1.281000,1.206629,2.099978
max,8.184574,8.280238,1.850462,11.204482,35.335929,34.000000,20.369918


In [19]:
# === Encode target labels ===
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
joblib.dump(label_encoder, 'label_encoder.pkl')
X = df.drop(columns=["label", "label_encoded"])
y = df["label_encoded"]

In [20]:
# === Train-test split (Stratified) ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(xgb.__version__)

3.0.2


In [21]:
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    eval_metric="mlogloss",
    use_label_encoder=False,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='exact'
)

# Train without early stopping
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

[0]	validation_0-mlogloss:1.56839
[1]	validation_0-mlogloss:1.40341
[2]	validation_0-mlogloss:1.26361
[3]	validation_0-mlogloss:1.15285
[4]	validation_0-mlogloss:1.05774
[5]	validation_0-mlogloss:0.97264
[6]	validation_0-mlogloss:0.90532
[7]	validation_0-mlogloss:0.84087
[8]	validation_0-mlogloss:0.78642
[9]	validation_0-mlogloss:0.73679
[10]	validation_0-mlogloss:0.69018
[11]	validation_0-mlogloss:0.65091
[12]	validation_0-mlogloss:0.61333
[13]	validation_0-mlogloss:0.57882
[14]	validation_0-mlogloss:0.54914
[15]	validation_0-mlogloss:0.52069
[16]	validation_0-mlogloss:0.49487
[17]	validation_0-mlogloss:0.47119
[18]	validation_0-mlogloss:0.45025
[19]	validation_0-mlogloss:0.42921
[20]	validation_0-mlogloss:0.41165
[21]	validation_0-mlogloss:0.39449
[22]	validation_0-mlogloss:0.37959
[23]	validation_0-mlogloss:0.36709
[24]	validation_0-mlogloss:0.35241
[25]	validation_0-mlogloss:0.34190
[26]	validation_0-mlogloss:0.33067
[27]	validation_0-mlogloss:0.31910
[28]	validation_0-mlogloss:0.3

c:\Users\msaad\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [16:51:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[33]	validation_0-mlogloss:0.26630
[34]	validation_0-mlogloss:0.26031
[35]	validation_0-mlogloss:0.25401
[36]	validation_0-mlogloss:0.24913
[37]	validation_0-mlogloss:0.24481
[38]	validation_0-mlogloss:0.23980
[39]	validation_0-mlogloss:0.23578
[40]	validation_0-mlogloss:0.23260
[41]	validation_0-mlogloss:0.22952
[42]	validation_0-mlogloss:0.22510
[43]	validation_0-mlogloss:0.22237
[44]	validation_0-mlogloss:0.21789
[45]	validation_0-mlogloss:0.21479
[46]	validation_0-mlogloss:0.21219
[47]	validation_0-mlogloss:0.20909
[48]	validation_0-mlogloss:0.20708
[49]	validation_0-mlogloss:0.20483
[50]	validation_0-mlogloss:0.20297
[51]	validation_0-mlogloss:0.20063
[52]	validation_0-mlogloss:0.19791
[53]	validation_0-mlogloss:0.19586
[54]	validation_0-mlogloss:0.19437
[55]	validation_0-mlogloss:0.19219
[56]	validation_0-mlogloss:0.19129
[57]	validation_0-mlogloss:0.18989
[58]	validation_0-mlogloss:0.18882
[59]	validation_0-mlogloss:0.18834
[60]	validation_0-mlogloss:0.18759
[61]	validation_0-ml

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [22]:
# === Predict and evaluate ===
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)  # Confidence scores

# === Display results ===
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

   antivirus       0.94      1.00      0.97        16
         cpu       0.86      0.43      0.57        14
      memory       0.91      0.94      0.93        34
     netproc       0.98      1.00      0.99        40
      normal       0.95      0.98      0.96       252
         tcp       0.96      0.90      0.93        50

    accuracy                           0.95       406
   macro avg       0.93      0.87      0.89       406
weighted avg       0.95      0.95      0.94       406

Confusion Matrix:
[[ 16   0   0   0   0   0]
 [  1   6   0   0   7   0]
 [  0   0  32   0   1   1]
 [  0   0   0  40   0   0]
 [  0   1   3   1 246   1]
 [  0   0   0   0   5  45]]


In [23]:
# === Example: Print prediction with confidence ===
for i in range(5):
    probs = y_proba[i]
    pred_class = np.argmax(probs)
    conf = probs[pred_class]
    print(f"Prediction: {label_encoder.inverse_transform([pred_class])[0]}, Confidence: {conf:.2f}")
    
model.save_model("xgb_bottleneck_model.json")  # or .bin

Prediction: normal, Confidence: 0.99
Prediction: memory, Confidence: 0.63
Prediction: normal, Confidence: 0.99
Prediction: normal, Confidence: 0.94
Prediction: normal, Confidence: 1.00
